<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/evaluation/04_run_colab_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# Phase 8: E2E Bass Transcription Pipeline Benchmark (Colab)
# [Cell 1] 환경 설정, GPU 할당 검증, 저장소 동기화 및 무결성 점검
# ==============================================================================

import os
import sys
import shutil
import importlib.util
import subprocess
from google.colab import drive
import torch
from datetime import datetime

# [사용자 환경 변수 세팅] - 본인의 드라이브 경로로 맞춰주세요.
DRIVE_DATASET_DIR = "/content/drive/MyDrive/Bass_separator/dataset"
DRIVE_ZIP_PATH = f"{DRIVE_DATASET_DIR}/slakh_test.zip"

print("📂 구글 드라이브 마운트 중...")
drive.mount('/content/drive')

if not torch.cuda.is_available():
    raise SystemError("❌ GPU가 할당되지 않았습니다. 상단 메뉴 [런타임] -> [런타임 유형 변경]에서 T4 GPU를 선택하십시오.")
print(f"✅ GPU 활성화됨: {torch.cuda.get_device_name(0)}")

print("\n📦 저장소 클론 및 작업 공간 초기화 중...")
!rm -rf /content/Bass-separator
!git clone https://github.com/sjkim-audio/Bass-separator.git /content/Bass-separator

if "/content/Bass-separator" not in sys.path:
    sys.path.append("/content/Bass-separator")
%cd /content/Bass-separator

print("\n🔧 [시스템] 필수 도구 확인 중...")
if shutil.which("ffmpeg") is None:
    print("⚠️ FFmpeg가 없습니다. apt-get으로 설치합니다...")
    !apt-get update -qq
    !apt-get install -y ffmpeg -qq
else:
    print("✅ FFmpeg가 이미 설치되어 있습니다.")

print("\n🐍 [파이썬] 라이브러리 설치 중...")
!pip install -q -r requirements.txt
!pip install -q mir_eval museval pretty_midi

print("\n🏥 설치 무결성 점검 (Health Check)...")
critical_libs = ["demucs", "torchaudio", "librosa", "museval", "mir_eval"]
missing = [lib for lib in critical_libs if importlib.util.find_spec(lib) is None]

if not missing:
    print("✅ 필수 라이브러리가 모두 정상적으로 준비되었습니다!")
else:
    raise ImportError(f"❌ 다음 라이브러리가 누락되었습니다: {', '.join(missing)}")

📂 구글 드라이브 마운트 중...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ GPU 활성화됨: Tesla T4

📦 저장소 클론 및 작업 공간 초기화 중...
Cloning into '/content/Bass-separator'...
remote: Enumerating objects: 2185, done.
remote: Counting objects: 100% (321/321), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 2185 (delta 260), reused 196 (delta 186), pack-reused 1864 (from 1)
Receiving objects: 100% (2185/2185), 305.16 MiB | 26.81 MiB/s, done.
Resolving deltas: 100% (1357/1357), done.
/content/Bass-separator

🔧 [시스템] 필수 도구 확인 중...
✅ FFmpeg가 이미 설치되어 있습니다.

🐍 [파이썬] 라이브러리 설치 중...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 352.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [ ]:
# ==============================================================================
# [Cell 2] 데이터셋 I/O 샌드박싱 및 압축 해제
# ==============================================================================

os.makedirs("./slakh_processed/test", exist_ok=True)

print("🗜️ 데이터셋 압축 해제 중 (수 분 소요될 수 있습니다)...")
!unzip -q {DRIVE_ZIP_PATH} -d ./slakh_processed/test

print("\n✅ 추출된 유효 트랙 수:")
!ls -1 ./slakh_processed/test | wc -l

🗜️ 데이터셋 압축 해제 중 (수 분 소요될 수 있습니다)...

✅ 추출된 유효 트랙 수:
130


In [ ]:
# ==============================================================================
# [Cell 3] 벤치마크 1: 채보 알고리즘 단독 평가 (Isolated Mode)
# ==============================================================================
current_date = datetime.now().strftime("%Y%m%d")
exp_id_isolated = f"Colab_Phase8_Isolated_{current_date}"

print(f"🚀 [1/2] 순수 미디 채보 성능 평가 시작 (ID: {exp_id_isolated})...")

!PYTHONPATH=/content/Bass-separator python -m src.evaluation.run_batch_eval \
    --test_dir ./slakh_processed/test \
    --isolated True \
    --onset_tolerance 0.1 \
    --exp_id {exp_id_isolated}

In [ ]:
# ==============================================================================
# [Cell 4] 벤치마크 2: 전체 파이프라인 성능 평가 (E2E Mode)
# ==============================================================================
current_date = datetime.now().strftime("%Y%m%d")
exp_id_e2e = f"Colab_Phase8_E2E_{current_date}"

print(f"🚀 [2/2] E2E 전체 파이프라인 평가 시작 (ID: {exp_id_e2e})...")

!PYTHONPATH=/content/Bass-separator python -m src.evaluation.run_batch_eval \
    --test_dir ./slakh_processed/test \
    --isolated False \
    --onset_tolerance 0.1 \
    --exp_id {exp_id_e2e}

In [ ]:
# ==============================================================================
# [Cell 5] 평가 결과 영구 백업
# ==============================================================================
# Colab 세션 종료 전, results 폴더에 생성된 모든 벤치마크 결과를 드라이브로 복사합니다.

print(f"💾 결과 파일을 {DRIVE_DATASET_DIR} 경로로 통합 백업합니다...")

# results 폴더 내의 모든 json 파일을 구글 드라이브로 일괄 복사
!cp ./results/*_batch_results.json {DRIVE_DATASET_DIR}

print("✅ 최종 평가 리포트 백업 완료.")